In [11]:
import ast
import json
import re
import tarfile
import os
import numpy as np
from pathlib import Path
from collections import defaultdict
from data_loader import TwinsDataLoader, LalondeDataLoader, ACSDataLoader, IHDPDataLoader, WalmartDataLoader
from experiments import prob_dict, largest_data_transformations
from search_methods.probe_ATE_search import ProbManager

In [ ]:
df_twins = TwinsDataLoader().load_data()
df_lalonde = LalondeDataLoader().load_data()
df_acs = ACSDataLoader().load_data()
df_IHDP = IHDPDataLoader().load_data()
df_walmart = WalmartDataLoader().load_data()

common_causes_twins = df_twins.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_lalonde = df_lalonde.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_acs = df_acs.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_ihdp = df_IHDP.columns.difference(["treatment", "outcome"], sort=False).tolist()
common_causes_walmart = df_walmart.columns.difference(["treatment", "outcome"], sort=False).tolist()

In [5]:
PROJECT_ROOT = r"C:\Users\daniel\PycharmProjects\ATE-shifting"

# sequence prob

In [ ]:
def get_prob(sequence, trans_dict, op_probs, common_causes):
    pm = ProbManager([func_name for func_name, func in trans_dict.items()], common_causes, op_probs)
    return pm.get_sequence_probability(sequence)

seq =(('bin_equal_width_5', 'sqft_basement'), ('bin_equal_frequency_2', 'grade'), ('bin_equal_frequency_2', 'bathrooms'), ('bin_equal_frequency_5', 'sqft_living'), ('bin_equal_frequency_2', 'sqft_lot15'), ('bin_equal_frequency_2', 'yr_built'), ('bin_equal_frequency_2', 'sqft_living15'))

prob = get_prob(seq, largest_data_transformations, prob_dict, common_causes_walmart)
print(prob)

# parse standalone runs

In [2]:
##########
# PARSE EXP RUN RESULTS
##########

def iter_out_files(path):
    """
    Yield (filename, content) from either:
      - a .tar.gz/.tgz/.tar archive
      - a directory containing extracted files
    Accepts both .out and .txt files.
    """
    path = Path(path)
    # Define acceptable extensions as a tuple
    valid_extensions = (".out", ".txt")

    if path.is_dir():
        # Walk recursively through extracted directory
        for file in path.rglob("*"):
            if file.is_file() and file.suffix.lower() in valid_extensions:
                yield file.name, file.read_text(errors="ignore")

    elif tarfile.is_tarfile(path):
        with tarfile.open(path, "r:*") as tar:   # supports tar, gz, bz2, xz, etc.
            for member in tar.getmembers():
                # Check if it is a file and ends with .out or .txt
                if not member.isfile() or not member.name.lower().endswith(valid_extensions):
                    continue

                f = tar.extractfile(member)
                if f is not None:
                    yield member.name, f.read().decode("utf-8", errors="ignore")

    else:
        raise ValueError(f"'{path}' is neither a directory nor a tar archive.")


def print_full_raw_logs(source):
    # Grouping: exp_name -> exp_type -> run_number -> full_raw_text
    grouped_data = defaultdict(lambda: defaultdict(dict))

    # Regex patterns for grouping and metric extraction
    start_pattern = re.compile(r"Starting\s+(?P<exp>\w+),\s+type=(?P<type>\w+),\s+run\s+(?P<run>\d+)/3")

    time_pat = re.compile(r"Execution time:\s*([\d\.]+)\s*(?:seconds|sec)?", re.IGNORECASE)
    popped_pat = re.compile(r"popped\s+(\d+)\s+from Q", re.IGNORECASE)
    checked_pat1 = re.compile(r"checked\s+(\d+)\s+combinations", re.IGNORECASE)
    checked_pat2 = re.compile(r"checked:\n(\d+)", re.IGNORECASE)
    seq_pat = re.compile(r"(?:sequence is:|Most probable sequence:)\s*(.*)", re.IGNORECASE)
    ate_now_pat = re.compile(r"ATE now is:\s*([\d\.]+)", re.IGNORECASE)

    prob_pat = re.compile(r"((?:\([^)]+\))?\s*probability of this sequence is:\s*[\d\.e\-]+)", re.IGNORECASE)
    uncertainty_pat = re.compile(r"uncertainty:\s*\n\s*(\{.*?\})", re.IGNORECASE)

    # Works for both archives and folders
    for filename, content in iter_out_files(source):
        match = start_pattern.search(content)
        if match:
            exp = match.group("exp")
            exp_type = match.group("type")
            run = match.group("run")
            grouped_data[exp][exp_type][run] = content

    if not grouped_data:
        print("No matching .out files found.")
        return

    metrics_order = ['Status', 'Sequence', 'ATE Now', 'Probability',
                     'Uncertainty', 'Time', 'Popped', 'Checked']

    for exp, types in grouped_data.items():
        for exp_type, runs in types.items():
            print("~" * 90)
            print(f"{exp} | {exp_type}")
            print("-" * 90)

            run_metrics = {}
            sorted_run_nums = sorted(runs.keys())

            for run_num in sorted_run_nums:
                raw_text = runs[run_num]
                m = {}

                m['Status'] = "TIMEOUT" if "*** TIMED OUT!! ***" in raw_text else "FINISHED"

                t_match = time_pat.search(raw_text)
                if t_match:
                    time_val = float(t_match.group(1))
                    m['Time'] = f"{time_val:.3f}s"
                    m['Raw_Time'] = time_val
                else:
                    m['Time'] = "N/A"
                    m['Raw_Time'] = None

                pop_match = popped_pat.search(raw_text)
                m['Popped'] = pop_match.group(1) if pop_match else "N/A"

                check_match = checked_pat1.search(raw_text) or checked_pat2.search(raw_text)
                m['Checked'] = check_match.group(1) if check_match else "N/A"

                seq_match = seq_pat.search(raw_text)
                m['Sequence'] = seq_match.group(1).strip() if seq_match else "N/A"

                ate_n_match = ate_now_pat.search(raw_text)
                m['ATE Now'] = f"{float(ate_n_match.group(1)):.6f}" if ate_n_match else "N/A"

                prob_matches = prob_pat.findall(raw_text)
                m['Probability'] = " / ".join(p.strip() for p in prob_matches) if prob_matches else "N/A"

                unc_match = uncertainty_pat.search(raw_text)
                if unc_match:
                    clean_unc = (
                        unc_match.group(1)
                        .replace("np.float64(", "")
                        .replace("np.True_", "True")
                        .replace("np.False_", "False")
                        .replace(")", "")
                    )
                    m['Uncertainty'] = clean_unc
                else:
                    m['Uncertainty'] = "N/A"

                run_metrics[run_num] = m

            for key in metrics_order:
                values = [run_metrics[r][key] for r in sorted_run_nums]

                if all(v == "N/A" for v in values):
                    continue

                unique_vals = set(values)

                if key == 'Time':
                    valid_times = [
                        run_metrics[r]['Raw_Time']
                        for r in sorted_run_nums
                        if run_metrics[r]['Raw_Time'] is not None
                    ]
                    mean_suffix = (
                        f" (Mean: {sum(valid_times)/len(valid_times):.3f}s)"
                        if valid_times else ""
                    )

                    if len(unique_vals) == 1:
                        print(f"  • {key:<12}: {values[0]}{mean_suffix}")
                    else:
                        run_strings = [
                            f"Run {r}: {run_metrics[r][key]}"
                            for r in sorted_run_nums
                        ]
                        print(f"  • {key:<12}: {' | '.join(run_strings)}{mean_suffix}")
                else:
                    if len(unique_vals) == 1:
                        print(f"  • {key:<12}: {values[0]}")
                    else:
                        run_strings = [
                            f"Run {r}: {run_metrics[r][key]}"
                            for r in sorted_run_nums
                        ]
                        print(f"  • {key:<12}: {' | '.join(run_strings)}")

            print()


In [ ]:
# print_full_raw_logs("acs_twins_run_28_05.tar.gz")
# print_full_raw_logs("exp_walmart.tar.gz")
# print_full_raw_logs("scale_exp")
# print_full_raw_logs("exp_27_28_long_TO.tar.gz")
print_full_raw_logs(os.path.join(PROJECT_ROOT, "exp_basic_larger.tar.gz"))

# Parse standalone \ ablation runs | save for graphs.py

In [ ]:

##########
# PARSE scale_exp RUN RESULTS | exp of type EXP<int>
##########

def parse_standalone_content(content):
    # Match: "Starting EXP27, type=probe, run 1/3"
    meta_match = re.search(r"Starting\s+EXP([\d.]+),\s*type=([\w.-]+).*?run\s+(\d+)", content, re.IGNORECASE)
    if not meta_match:
        return None, None, None, None

    exp_id = meta_match.group(1)
    exp_type = meta_match.group(2)
    run_num = int(meta_match.group(3))

    # We only want whole numbers in this script
    if "." in exp_id:
        return None, None, None, None

    status = "TIMEOUT" if "TIME OUT" in content else "FINISHED"

    time_match = re.search(r"Execution time:\s*([\d.]+)\s*sec", content)
    exec_time = float(time_match.group(1)) if time_match and status == "FINISHED" else None

    distances = []
    dist_match = re.search(r"distances from ATE \(with time\):\s*\n(\[.*?\])", content, re.DOTALL)
    if dist_match and status == "FINISHED":
        list_str = re.sub(r"np\.float64\((.*?)\)", r"\1", dist_match.group(1))
        try:
            distances = ast.literal_eval(list_str)
        except Exception:
            pass

    return exp_id, exp_type, run_num, {"status": status, "time": exec_time, "distances": distances}

def process_standalone(folder_path):
    # Hierarchy: { "27": { "probe": { 1: data, 2: data, 3: data } } }
    raw_results = defaultdict(lambda: defaultdict(dict))

    dir_path = Path(folder_path)
    print(f"Reading folder '{dir_path}' for Standalone Experiments...")

    # Look for files ending in either .out or .err inside the folder and subfolders
    for file_path in dir_path.rglob("*"):
        if file_path.is_file() and file_path.suffix in (".out", ".err"):
            # Read and decode the file contents directly from disk
            content = file_path.read_text(encoding="utf-8", errors="ignore")

            exp_id, exp_type, run_num, run_data = parse_standalone_content(
                content
            )

            if exp_id:
                raw_results[exp_id][exp_type][run_num] = run_data

    final_output = {}
    for exp_id, types_dict in raw_results.items():
        exp_key = f"EXP{exp_id}"
        final_output[exp_key] = {}

        for exp_type, runs_dict in types_dict.items():
            # Ensure order: run 1, run 2, run 3
            sorted_runs = [
                runs_dict.get(
                    i, {"status": "MISSING", "time": None, "distances": []}
                )
                for i in range(1, 4)
            ]

            exec_times = [r["time"] for r in sorted_runs if r["time"] is not None]
            mean_time = float(np.mean(exec_times)) if exec_times else None
            all_distances = [r["distances"] for r in sorted_runs]
            statuses = [r["status"] for r in sorted_runs]

            final_output[exp_key][exp_type] = {
                "run_statuses": statuses,
                "execution_times": exec_times,
                "mean_execution_time": mean_time,
                "distances_per_run": all_distances,
            }

    return final_output

In [ ]:
TAR_FILE_PATH = os.path.join(PROJECT_ROOT, "scale_exp")
results = process_standalone(TAR_FILE_PATH)
output_file = os.path.join(PROJECT_ROOT, "standalone_results.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)
print("Saved standalone_results.json")

In [ ]:
##########
# PARSE ablation RUN RESULTS | exp of type EXP<int>
##########
results_ablation_twins = process_standalone(os.path.join(PROJECT_ROOT, 'ablation_twins (1)'))
with open(os.path.join(PROJECT_ROOT, "ablation_twins.json"), "w") as f:
    json.dump(results_ablation_twins, f, indent=4)
print("Saved ablation twins.json")

results_ablation_acs = process_standalone(os.path.join(PROJECT_ROOT, 'ablation_acs'))
with open(os.path.join(PROJECT_ROOT, "ablation_acs.json"), "w") as f:
    json.dump(results_ablation_acs, f, indent=4)
print("Saved ablation acs.json")

results_ablation_walmart = process_standalone(os.path.join(PROJECT_ROOT, 'ablation_walmart'))
with open(os.path.join(PROJECT_ROOT, "ablation_walmart.json"), "w") as f:
    json.dump(results_ablation_walmart, f, indent=4)
print("Saved ablation walmart.json")

# Parse parametrized runs | save for graphs.py

In [ ]:
##########
# PARSE scale_exp RUN RESULTS | exp of type EXP<int>.<int>
##########

from pathlib import Path
import json
from collections import defaultdict

# Change this value to whatever placeholder you prefer (e.g., -1, 99999, or "TIMEOUT")
TIMEOUT_VAL = 14400


def parse_dotted_content(content):
    # Match: "Starting EXP30.1, type=probe, run 1/3"
    meta_match = re.search(
        r"Starting\s+EXP([\d.]+),\s*type=([\w.-]+).*?run\s+(\d+)",
        content,
        re.IGNORECASE,
    )
    if not meta_match:
        return None, None, None, None, None

    full_exp_id = meta_match.group(1)
    exp_type = meta_match.group(2)
    run_num = int(meta_match.group(3))

    # We ONLY want dotted numbers in this script
    if "." not in full_exp_id:
        return None, None, None, None, None

    parent_id = full_exp_id.split(".")[0]  # "30.1" -> "30"

    if "TIME OUT" in content:
        return parent_id, full_exp_id, exp_type, run_num, TIMEOUT_VAL

    time_match = re.search(r"Execution time:\s*([\d.]+)\s*sec", content)
    exec_time = float(time_match.group(1)) if time_match else TIMEOUT_VAL

    return parent_id, full_exp_id, exp_type, run_num, exec_time


def process_dotted(folder_path):
    # Hierarchy: { "30": { "probe": { "30.1": {1: time, 2: time, 3: time}, "30.2": {...} } } }
    raw_results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

    dir_path = Path(folder_path)
    print(f"Reading folder '{dir_path}' for Dotted Sub-experiments...")

    # Look for files ending in either .out or .err inside the folder and subfolders
    for file_path in dir_path.rglob("*"):
        if file_path.is_file() and file_path.suffix in (".out", ".err"):
            # Read and decode the file contents directly from disk
            content = file_path.read_text(encoding="utf-8", errors="ignore")

            parent_id, full_id, exp_type, run_num, exec_time = (
                parse_dotted_content(content)
            )

            if parent_id:
                raw_results[parent_id][exp_type][full_id][run_num] = exec_time

    final_output = {}
    for parent_id, types_dict in raw_results.items():
        parent_key = f"EXP{parent_id}"
        final_output[parent_key] = {}

        for exp_type, sub_exps in types_dict.items():
            # Sort sub-experiments numerically so 30.1 comes before 30.2
            sorted_sub_exp_keys = sorted(sub_exps.keys(), key=lambda x: float(x))

            type_run_lists = []
            for sub_key in sorted_sub_exp_keys:
                runs_dict = sub_exps[sub_key]
                # Keep exactly 3 entries. If a run doesn't exist or timed out, use TIMEOUT_VAL
                times = [runs_dict.get(i, TIMEOUT_VAL) for i in range(1, 4)]
                type_run_lists.append(times)

            final_output[parent_key][exp_type] = type_run_lists

    return final_output


In [ ]:
TAR_FILE_PATH = os.path.join(PROJECT_ROOT, "scale_exp")
results = process_dotted(TAR_FILE_PATH)

with open(os.path.join(PROJECT_ROOT, "dotted_results.json"), "w") as f:
    json.dump(results, f, indent=4)
print("Saved dotted_results.json")

# Parse F experiments

In [ ]:
##########
# PARSE F_exp RUN RESULTS | exp of type EXP<int>.<int>
##########
import json


def parse_and_sort_experiment_archive(tar_path):
    # Example:
    # Starting EXP41.1, type=fprobe, run 1/3
    header_regex = re.compile(
        r"Starting EXP(\d+)\.(\d+),\s*type=([^,]+),\s*run (\d+)/(\d+)"
    )

    time_regex = re.compile(r"Execution time:\s*([\d\.]+)\s*sec")

    raw_results = {}

    # Read files directly from tar.gz
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():

            # Only process .out files
            if not member.isfile() or not member.name.endswith(".out"):
                continue

            f = tar.extractfile(member)
            if f is None:
                continue

            lines = f.read().decode("utf-8", errors="replace").splitlines(True)

            if not lines:
                continue

            first_line = lines[0].strip()

            match = header_regex.search(first_line)
            if not match:
                continue

            major_exp = match.group(1)
            sub_exp = match.group(2)
            run_type = match.group(3)
            run_idx = int(match.group(4)) - 1

            sub_exp_key = f"{major_exp}.{sub_exp}"

            # Create structure:
            # EXP -> sub EXP -> type -> [run1, run2, run3]
            raw_results.setdefault(major_exp, {})
            raw_results[major_exp].setdefault(sub_exp_key, {})
            raw_results[major_exp][sub_exp_key].setdefault(
                run_type,
                [None, None, None]
            )

            file_content = "".join(lines)

            if "TIMED OUT!" in file_content:
                run_result = "TIMEOUT"

            elif "FINISHED" in file_content:
                time_match = time_regex.search(file_content)
                run_result = (
                    float(time_match.group(1))
                    if time_match
                    else "FINISHED"
                )

            else:
                run_result = "UNKNOWN"

            if 0 <= run_idx < 3:
                raw_results[major_exp][sub_exp_key][run_type][run_idx] = run_result


    # Sort output:
    # EXP number -> sub experiment -> type alphabetically
    perfectly_sorted = {}

    for major in sorted(raw_results.keys(), key=int):
        perfectly_sorted[major] = {}

        sorted_subs = sorted(
            raw_results[major].keys(),
            key=lambda x: [int(num) for num in x.split(".")]
        )

        for sub in sorted_subs:
            perfectly_sorted[major][sub] = {}

            for run_type in sorted(raw_results[major][sub].keys()):
                perfectly_sorted[major][sub][run_type] = (
                    raw_results[major][sub][run_type]
                )

    return perfectly_sorted


In [ ]:
archive = os.path.join(PROJECT_ROOT, "exp_F_2.tar.gz")
exp_data = parse_and_sort_experiment_archive(archive)

print(json.dumps(exp_data, indent=4))

with open(os.path.join(PROJECT_ROOT, "F_exp.json"), "w") as f:
    json.dump(exp_data, f, indent=4)

print("Saved F_exp.json")


# TODO: was relevant for prior runs. may not be relevant
archive = os.path.join(PROJECT_ROOT, "exp_f_walmart_no_rest.tar.gz")
exp_data = parse_and_sort_experiment_archive(archive)

print(json.dumps(exp_data, indent=4))

with open(os.path.join(PROJECT_ROOT, "F_exp_walmart_no_restart.json"), "w") as f:
    json.dump(exp_data, f, indent=4)

print("Saved exp_f_walmart_no_rest.json")